Auditoria Gran Encuesta Integrada de Hogares

4 Modulos
 . CGSE
 . FT
 . OCUPADOS
 . DESEMPLEADOS

In [0]:
#Leyendo la tabla de caracteristicas generales
cgse = spark.read.format("delta").table("workspace.electiva.cgse")


In [0]:
#Convertir el DataFrame de spark a Pandas
cgse = cgse.toPandas()


In [0]:
cgse

In [0]:
cgse.display()

Exploración de modulos y calculos base

**Funciones basicas de **pandas****

In [0]:
# Filtrando varias columnas de la tabla nivel de columna
cgse[['PERIODO','FEX_C18','P6090']]

In [0]:
#Filtro a nivel de fila, filtrar solo personas donde la variable DPTO=63
cgse[cgse['DPTO']==63]

In [0]:
# Filtrando a nivel de columna
cgse['PERIODO']

#Aplicando operaciones
cgse['PERIODO'].sum()
cgse['PERIODO'].max()
cgse['PERIODO'].min()
cgse['PERIODO'].mean()
cgse['PERIODO'].std()
cgse['PERIODO'].var()
cgse['PERIODO'].median()
cgse['PERIODO'].mode()

In [0]:
cgse["P3271"]. unique()

# 1. **Población **total****

In [0]:
cgse
#PT = cgse["FEX_C18"]. sum()

In [0]:
#Primeros n registros de la tabla
cgse.head(20)

In [0]:
#Leyendo la tabla de caracteristicas generales
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
ft = spark.read.format("delta").table("workspace.electiva.ft")
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados")

# Convertir el DataFrame de Spark a Pandas
cgse = cgse.toPandas()
desempleados=desempleados.toPandas()
ft=ft.toPandas()
ocupados=ocupados.toPandas()

Cantidad de personas desempleadas

In [0]:
D = desempleados[desempleados["DSI"]==1]["FEX_C18"].sum()
D

Como calcular la PEA

In [0]:
PEA = ft[ft["FT"]==1]["FEX_C18"].sum()
PEA

Calculando la tasa de desempleo

In [0]:
TD = D/PEA*100
print(f"la tasa de desempleo en colombia para junio 2026 es de {TD} % ")

In [0]:
#Actividad con la tasa global de participacion
PET = ft[ft["PET"]==1]["FEX_C18"].sum()
TGP=PEA/PET*100
print(f"La tasa global de participacion en colombia para junio 2026 es de {TGP} % ")

In [0]:
#Actividad con la tasa de ocupacion
O = ocupados[ocupados["OCI"]==1]["FEX_C18"].sum()
TO=O/PEA*100
TO=O/PET*100
print(f"La tasa de ocupacion en colombia para junio 2026 es de {TO} % ")


In [0]:
ocupados["RAMA2D_R4"].unique()

In [0]:
#Calculando cuantas personas trabajan en el sector de alojamiento y servicios de comida en colombia a junio de 2026
ocupados[(ocupados['RAMA2D_R4']==55) | (ocupados['RAMA2D_R4']==56) ]['FEX_C18'].sum()
# En colombia en el sector de alojamiento y servicios de comida trabajan 2.057.354 personas a junio de 2026

In [0]:
#Calculando cuantas personas trabajan en el sector de construccion en colombia a junio de 2026
ocupados[(ocupados['RAMA2D_R4']==41) | (ocupados['RAMA2D_R4']==42) | (ocupados['RAMA2D_R4']==43)]['FEX_C18'].sum()
# En colombia en el sector de construccion trabajan 1.678.378 personas

In [0]:
#Calculando cuantas personas trabajan en el sector de actividades inmobiliarias en colombia a junio de 2026
ocupados[ocupados['RAMA2D_R4']==68]['FEX_C18'].sum()
# En colombia en el sector de actividades inmobiliarias trabajan 302.935 personas a junio de 2026

Calculo de indicadores laborales para el departamento del Quindío Junio 2026 segmentado por genero.

Vamos a responder a la pregunta :

Cual es la tasa de desempleo en hombres en el quindio para junio 2026
Cual es la tasa de desempleo en mujeres en el quindio para junio 2026

In [0]:


cgse[(cgse['DIRECTORIO']==8637743) & (cgse['SECUENCIA_P']==1) & (cgse['ORDEN']==1) ][['DIRECTORIO','SECUENCIA_P','ORDEN','P3271']]

Necesitamos traernos la variable p3271 del modulo de cgse, y añadirla al modulo de desempleados para poder calcular tasa de desempleo en hombres y mujeres del quindío

In [0]:
#Primero hay que anexar la p3271 a la tabla de ft
ft = pd.merge(ft, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']],
 on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

In [0]:
## calcular PEA de mujeres y hombres en el quindio

# 1.PEA de mujeres en el quindio
PEAMQ = ft[(ft['FT']==1) & (ft['DPTO']==63) & (ft['P3271_x']==2)]['FEX_C18'].sum()
# 2.PEA de hombres en el quindio
PEAHQ = ft[(ft['FT']==1) & (ft['DPTO']==63) & (ft['P3271_x']==1)]['FEX_C18'].sum()

print(f"en el quindio hay {PEAMQ} Mujeres en el PEA")
print(f"en el quindio hay {PEAHQ} Hombres en el PEA")

In [0]:
#Por ultimo cuando ya tenemos cantidad de desempleados y PEA podemos calcular el porcentaje de desempleo

TDHQ = HDQ/PEAHQ *100
print(f"El porcentaje de desempleo masculino en el quindio es de {TDHQ}%")
TDMQ = MDQ/PEAMQ *100
print(f"El porcentaje de desempleo femenino en el quindio es de {TDMQ}%")